In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
# df = pd.read_csv("test.csv")
df_train = pd.read_csv("train.csv")

In [4]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

# 1. Chargement des données
df = pd.read_csv('train.csv')

# --- GESTION DU CODE POSTAL (DÉPARTEMENT) ---
# On prend les 2 premiers chiffres pour éviter d'avoir trop de colonnes
df['code_postal'] = df['code_postal'].astype(str).str.zfill(5).str[:2]

# 2. Préparation des variables
df['anciennete_vehicule'] = df['anciennete_vehicule'].fillna(df['anciennete_vehicule'].median())

numerical_features = ['bonus', 'duree_contrat', 'age_conducteur1', 'anciennete_permis1', 'prix_vehicule','vitesse_vehicule','cylindre_vehicule','anciennete_vehicule','din_vehicule','poids_vehicule']
categorical_features = ['type_contrat', 'utilisation', 'essence_vehicule', 'code_postal','marque_vehicule']

# --- MODÈLE 1 : PROBABILITÉ (FRÉQUENCE) ---
df['target_sinistre'] = (df['nombre_sinistres'] > 0).astype(int)
X = df[numerical_features + categorical_features]
y_freq = df['target_sinistre']

# Pipeline unique pour la fréquence
model_freq = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])),
    ('classifier', LogisticRegression(max_iter=1000))
])

model_freq.fit(X, y_freq)

# --- MODÈLE 2 : MONTANT (SÉVÉRITÉ) ---
# On entraîne uniquement sur ceux qui ont eu un sinistre
df_claims = df[df['montant_sinistre'] > 0].copy()
X_sev = df_claims[numerical_features + categorical_features]
y_sev = df_claims['montant_sinistre']

# Pipeline unique pour le montant (évite le bug des 108/109 features)
model_sev = Pipeline(steps=[
    ('preprocessor', ColumnTransformer(transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
    ])),
    ('regressor', LinearRegression())
])

model_sev.fit(X_sev, y_sev)

# --- APPLICATION DES PRÉDICTIONS ---
# On calcule tout sur le dataframe d'origine (ou un nouveau fichier)
df['probabilite'] = model_freq.predict_proba(X)[:, 1]
df['montant_potentiel'] = model_sev.predict(X)
df['cout_espere'] = df['probabilite'] * df['montant_potentiel']

print("Prédictions terminées avec succès !")
print(df[['id_contrat', 'code_postal', 'probabilite', 'cout_espere']].head())

Prédictions terminées avec succès !
      id_contrat code_postal  probabilite  cout_espere
0  A00000001-V01          36     0.058488    86.155270
1  A00000002-V01          92     0.123142   271.079644
2  A00000003-V01          92     0.045448    94.557986
3  A00000004-V01          78     0.033702    41.576777
4  A00000005-V01          38     0.064540    99.503294


c:\Users\CYTech Student\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [3, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [5]:
import pandas as pd

# 1. Charger le nouveau fichier
nouveau_df = pd.read_csv('test.csv')

# 2. Prétraitement identique (très important !)
# On remplit les valeurs manquantes comme pour l'entraînement
nouveau_df['anciennete_vehicule'] = nouveau_df['anciennete_vehicule'].fillna(nouveau_df['anciennete_vehicule'].median())

# 3. Sélectionner les colonnes utilisées par le modèle
# (Assurez-vous que numerical_features et categorical_features sont bien définis)
X_nouveau = nouveau_df[numerical_features + categorical_features]

# 4. Appliquer les prédictions
# Probabilité de sinistre (Modèle 1)
nouveau_df['probabilite'] = model_freq.predict_proba(X_nouveau)[:, 1]

# Montant estimé si sinistre il y a (Modèle 2)
nouveau_df['montant_potentiel'] = model_sev.predict(X_nouveau)

# Coût espéré (Prime pure)
nouveau_df['cout_espere'] = nouveau_df['probabilite'] * nouveau_df['montant_potentiel']

# 5. Sauvegarder les résultats
nouveau_df.to_csv('resultats_predictions.csv', index=False)
print("Les prédictions ont été sauvegardées dans 'resultats_predictions.csv'")

c:\Users\CYTech Student\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [3, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\CYTech Student\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [3, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Les prédictions ont été sauvegardées dans 'resultats_predictions.csv'


In [6]:
pp=pd.read_csv('resultats_predictions.csv')
df_final = pp[['index', 'cout_espere']]

# 3. Sauvegarder le résultat dans un nouveau fichier CSV
# index=False permet de ne pas rajouter une colonne de numérotation inutile
df_final.to_csv('resultat_final2.csv', index=False)

print("Le fichier filtré a été sauvegardé sous le nom 'resultat_final.csv'")

Le fichier filtré a été sauvegardé sous le nom 'resultat_final.csv'
